# 第6章 用 rocprof 找到慢在哪里

**操作手册** | 对照两个 vector add，只看 kernel 时间、工作划分和 stride 趋势

本手册对应文档：`docs/part1-profiling/chapter6/index.md`  
本手册对应代码：`code/part1-profiling/chapter6/`

---

## Goal

学会用 `rocprofv3` 定位慢 kernel，理解工作划分对性能的影响。具体目标：

1. 用 benchmark 确认两个实现的速度差距
2. 用 `rocprofv3 --kernel-trace` 找到慢在哪个 dispatch
3. 对比 coalesced 和 linecross 的 Grid Size、VGPR、SGPR
4. 扫描 stride 参数，观察性能趋势
5. 识别实验同时改变了哪些变量（地址排布、循环次数、Grid Size）

## Prerequisite

- 已完成第5章 benchmark 与可信计时
- ROCm 环境已激活（`source code/part1-profiling/activate-rocm.sh`）
- `rocprofv3` 可用（ROCm 7.13+）
- 理解 warmup、repeat、GPU event 计时

## Platform

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **hipcc**: 7.13.99004 / arch gfx1201

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，数字会有差异。

## Parameter

### vector_add_bench 参数

| 参数 | 含义 | 示例值 |
|------|------|--------|
| `--kernel` | 选择 coalesced 或 linecross | coalesced |
| `--size` | 处理元素个数 | 16777216 (16M) |
| `--block` | 每个 block 的线程数 | 256 |
| `--stride` | linecross 中每个 lane 负责的连续元素数 | 1, 32 |
| `--warmup` | 热身次数 | 20 |
| `--repeat` | 正式计时次数 | 100 |
| `--output-json` | 保存结果到 JSON | logs/result.json |

### rocprofv3 参数

| 参数 | 含义 |
|------|------|
| `--kernel-trace` | 收集 kernel dispatch 跟踪 |
| `-o <file>` | 输出文件路径 |
| `-f csv` | 输出格式为 CSV |

## Execution

### 步骤1：定位仓库根目录并进入工作目录

In [ ]:
import os
import subprocess
from pathlib import Path

# 定位仓库根目录
REPO_ROOT = Path.cwd().resolve().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
WORK_DIR = REPO_ROOT / "code/part1-profiling/chapter6"
os.chdir(WORK_DIR)

print(f"仓库根目录: {REPO_ROOT}")
print(f"工作目录: {WORK_DIR}")
print(f"当前目录: {Path.cwd()}")

### 步骤2：检测 GPU 架构

在编译前自动检测当前 GPU 架构（gfx1100/gfx1151/gfx1201），避免硬编码特定架构。

In [ ]:
# 检测 GPU 架构（gfx1100/gfx1151/gfx1201）
import subprocess

def detect_gpu_arch():
    """检测当前 GPU 架构，返回 gfxXXXX 字符串"""
    try:
        # 方法1: 尝试 rocminfo
        result = subprocess.run(
            ["rocminfo"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if 'Name:' in line and 'gfx' in line.lower():
                    # 提取 gfxXXXX
                    parts = line.split()
                    for part in parts:
                        if part.startswith('gfx'):
                            return part
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    
    try:
        # 方法2: 尝试 rocm-smi --showproductname
        result = subprocess.run(
            ["rocm-smi", "--showproductname"],
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            output = result.stdout.lower()
            # 根据产品名推断架构
            if '9070' in output or '9060' in output:
                return 'gfx1201'  # RDNA4
            elif '7900' in output or '7800' in output or '7700' in output:
                return 'gfx1100'  # RDNA3
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    
    # 默认回退到 gfx1201（9070XT baseline）
    print("⚠ 无法自动检测架构，使用默认值 gfx1201")
    return 'gfx1201'

# 检测并验证架构
GPU_ARCH = detect_gpu_arch()
print(f"检测到 GPU 架构: {GPU_ARCH}")

# 验证架构是否在支持列表中
SUPPORTED_ARCHS = ['gfx1100', 'gfx1151', 'gfx1201']
if GPU_ARCH not in SUPPORTED_ARCHS:
    print(f"⚠ 警告: {GPU_ARCH} 不在已验证列表 {SUPPORTED_ARCHS} 中")
    print(f"  继续使用 {GPU_ARCH}，但结果可能与文档基线不同")
else:
    print(f"✓ 架构 {GPU_ARCH} 已验证（支持 RDNA3/RDNA4）")


### 步骤3：编译 vector_add_bench

编译 HIP 程序，生成 benchmark 可执行文件：

In [ ]:
# 创建 logs 目录
logs_dir = WORK_DIR / "logs"
logs_dir.mkdir(exist_ok=True)

# 编译 vector_add.hip（使用检测到的架构）
source_file = WORK_DIR / "vector_add.hip"
output_binary = WORK_DIR / "vector_add_bench"

if source_file.exists():
    print(f"编译: {source_file.name}")
    compile_cmd = [
        "hipcc",
        f"--offload-arch={GPU_ARCH}",  # 使用检测到的架构
        "-O3",
        str(source_file),
        "-o",
        str(output_binary)
    ]
    print(f"编译命令: {' '.join(compile_cmd)}")
    result = subprocess.run(compile_cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"编译成功: {output_binary}")
    else:
        print(f"编译失败: {result.stderr}")
else:
    print(f"未找到源文件: {source_file}")


### 步骤4：运行 benchmark — coalesced 版本

In [ ]:
if output_binary.exists():
    print("运行 coalesced 版本:")
    cmd = [
        str(output_binary),
        "--kernel", "coalesced",
        "--size", "16777216",
        "--block", "256",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "coalesced_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print(f"未找到可执行文件: {output_binary}")

### 步骤5：运行 benchmark — linecross stride=1

In [ ]:
if output_binary.exists():
    print("运行 linecross stride=1:")
    cmd = [
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "1",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "linecross_stride1_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")

### 步骤6：运行 benchmark — linecross stride=32

In [ ]:
if output_binary.exists():
    print("运行 linecross stride=32:")
    cmd = [
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "32",
        "--warmup", "20",
        "--repeat", "100",
        "--output-json", str(logs_dir / "linecross_stride32_size16777216.json")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")

### 步骤7：检查 rocprofv3 可用性

在采集 kernel trace 前，先检查 `rocprofv3` 是否可用：

In [ ]:
# 检查 rocprofv3 是否可用
try:
    result = subprocess.run(
        ["rocprofv3", "--version"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        print("rocprofv3 可用:")
        print(result.stdout)
        rocprof_available = True
    else:
        print("rocprofv3 不可用")
        rocprof_available = False
except FileNotFoundError:
    print("rocprofv3 未找到，请确认 ROCm 环境已激活")
    rocprof_available = False

### 步骤7：采集 kernel trace（如果 rocprofv3 可用）

使用 `rocprofv3 --kernel-trace` 收集每次 kernel dispatch 的时间戳和配置：

In [ ]:
if rocprof_available and output_binary.exists():
    print("采集 coalesced kernel trace:")
    cmd = [
        "rocprofv3",
        "--kernel-trace",
        "-o", str(logs_dir / "final_kt_coalesced.csv"),
        "-f", "csv",
        "--",
        str(output_binary),
        "--kernel", "coalesced",
        "--size", "16777216",
        "--block", "256",
        "--warmup", "5",
        "--repeat", "10"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
    
    print("\n采集 linecross stride=32 kernel trace:")
    cmd = [
        "rocprofv3",
        "--kernel-trace",
        "-o", str(logs_dir / "final_kt_linecross32.csv"),
        "-f", "csv",
        "--",
        str(output_binary),
        "--kernel", "linecross",
        "--size", "16777216",
        "--block", "256",
        "--stride", "32",
        "--warmup", "5",
        "--repeat", "10"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(result.stdout)
    if result.returncode != 0:
        print(f"错误: {result.stderr}")
else:
    print("跳过 kernel trace 采集（rocprofv3 不可用或可执行文件不存在）")

### 步骤8：reference fallback — 加载已有证据

当 profiler 不可用时，加载仓库中已有的参考数据：

In [ ]:
import json

# reference fallback: 加载已有的实测数据
reference_data = {
    "platform": "Radeon RX 9070 XT (gfx1201) + ROCm 7.13",
    "measured": {
        "coalesced": {
            "min_time_ms": 0.334,
            "median_time_ms": 0.337,
            "effective_bw_gbps": 603,
            "grid_size": 16777216,
            "vgpr": 8,
            "sgpr": 128
        },
        "linecross_stride1": {
            "min_time_ms": 0.336,
            "median_time_ms": 0.338,
            "effective_bw_gbps": 599,
            "grid_size": 16777216,
            "vgpr": 16,
            "sgpr": 128
        },
        "linecross_stride32": {
            "min_time_ms": 2.25,
            "median_time_ms": 2.30,
            "effective_bw_gbps": 89.7,
            "grid_size": 524288,
            "vgpr": 16,
            "sgpr": 128
        }
    },
    "hypothesis": [
        "linecross stride=32 同时改变了地址排布、每线程循环次数和 Grid Size",
        "Grid Size 从 16M 降到 524K（1/32），说明线程工作划分确实变了",
        "VGPR/SGPR 分配相同（stride 是运行时参数），静态资源不是变量",
        "有效带宽从 603 GB/s 降到 89.7 GB/s（约 6.7 倍差距）"
    ]
}

print("=== Reference Data (measured on 9070XT + ROCm 7.13) ===")
print(json.dumps(reference_data, indent=2, ensure_ascii=False))

### 步骤9：stride 扫描（观察趋势）

扫描多个 stride 值，观察 linecross 实现的整体性能变化：

In [ ]:
if output_binary.exists():
    print("stride 扫描:")
    stride_values = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    
    for s in stride_values:
        cmd = [
            str(output_binary),
            "--kernel", "linecross",
            "--size", "16777216",
            "--block", "256",
            "--stride", str(s),
            "--warmup", "20",
            "--repeat", "50",
            "--output-json", str(logs_dir / f"linecross_s{s}.json")
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        # 只打印关键行
        for line in result.stdout.split('\n'):
            if 'stride' in line.lower() or 'time' in line.lower() or 'bandwidth' in line.lower():
                print(line)
else:
    print(f"未找到可执行文件: {output_binary}")
    print("\nreference fallback: stride 扫描参考数据")
    stride_reference = [
        (1, 0.338, 596),
        (8, 0.405, 497),
        (16, 1.65, 122),
        (32, 2.19, 92.1),
        (64, 4.86, 41.4),
        (256, 25.3, 7.95)
    ]
    print(f"{'stride':>6} | {'time_ms':>8} | {'eff_bw_gbps':>12} | {'vs_stride1':>10}")
    print("-" * 45)
    for s, t, bw in stride_reference:
        ratio = t / stride_reference[0][1]
        print(f"{s:>6} | {t:>8.3f} | {bw:>12.2f} | {ratio:>9.2f}x")

## Expected Output / Interpretation

### benchmark 预期输出

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

| kernel | stride | 最短时间 | 有效带宽 | 正确性 |
|--------|--------|----------|----------|--------|
| coalesced | - | 0.334 ms | 603 GB/s | OK |
| linecross | 1 | 0.336 ms | 599 GB/s | OK |
| linecross | 32 | 2.25 ms | 89.7 GB/s | OK |

**解读**：
- `linecross stride=1` 与 coalesced 时间接近（每个线程也只处理一个元素）
- `linecross stride=32` 慢约 6.7 倍，但同时改变了地址排布、循环次数和 Grid Size

### kernel trace 预期输出（如果 rocprofv3 可用）

CSV 文件中关键列：

| kernel | 单次最短时间 | 中位数 | Grid Size | VGPR | SGPR |
|--------|--------------|--------|-----------|------|------|
| kernel_coalesced | 329 μs | 330 μs | 16,777,216 | 8 | 128 |
| kernel_linecross (s=32) | 2202 μs | 2304 μs | 524,288 | 16 | 128 |

**解读**：
- kernel trace 的 329 μs 与 benchmark 的 0.334 ms 基本一致
- `kernel_linecross` 的 Grid Size 只有 coalesced 的 1/32
- VGPR/SGPR 分配相同（stride 是运行时参数）

### stride 扫描预期趋势

| stride | 相对 stride=1 耗时 |
|--------|--------------------|
| 1 | 1.00× |
| 8 | 1.20× |
| 16 | 4.89× |
| 32 | 6.47× |
| 64 | 14.4× |
| 256 | 74.9× |

**解读**：
- stride 整体越大，这个实现越慢
- 但一个参数同时改变了多个底层变量，这是**组合效果**
- 要单独验证访存合并，需要固定线程数和循环次数，只改索引公式

### 标签说明

- **measured**: 实测数据（来自 GPU event 或 rocprofv3）
- **reference**: 参考数据（当 profiler 不可用时的 fallback）
- **hypothesis**: 当前假设（需要后续公平对照验证）

## Pass Criteria

本章操作通过标准：

### 必须满足

1. ✅ 能编译并运行 `vector_add_bench`，输出 coalesced 和 linecross 的时间
2. ✅ 确认 `linecross stride=32` 明显慢于 coalesced（约 6.7 倍）
3. ✅ 理解 stride 同时改变了地址排布、循环次数和 Grid Size
4. ✅ 能解读 benchmark 输出：最短时间、有效带宽、正确性
5. ✅ 理解 reference fallback：当 profiler 不可用时，加载已有证据

### 推荐完成（如果 rocprofv3 可用）

6. ⭐ 采集 kernel trace，对比 Grid Size、VGPR、SGPR
7. ⭐ 验证 kernel trace 时间与 benchmark 时间一致
8. ⭐ 观察 stride 扫描曲线，识别整体趋势

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| hipcc 编译失败 | ROCm 未激活 | `source code/part1-profiling/activate-rocm.sh` |
| rocprofv3 不可用 | ROCm 版本 < 7.13 | 使用 reference fallback 数据 |
| 时间差异很大 | 后台负载 | 关闭其他 GPU 任务 |
| CSV 文件为空 | profiler 权限问题 | 检查 `/tmp` 写权限 |
| 正确性检查失败 | 输入规模或 stride 不匹配 | 检查命令行参数 |

---

## 延伸阅读

- [ROCprofiler 文档](https://rocm.docs.amd.com/projects/rocprofiler/en/latest/)
- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)
- [GPUOpen: Memory Coalescing](https://gpuopen.com/learn/gcn-memory-coalescing/)

**下一章**: [第7章 读懂 Roofline 图](./chapter7.ipynb)